# 09 -- Macro Daily & Monthly WRDS Data Collection

## Purpose
Downloads all non-stock-specific (market-level and macroeconomic) factor data from WRDS, saved as two flat parquet files: one daily, one monthly. These datasets require no PERMNO filtering.

## Source
WRDS via the `wrds` Python library, authenticated with username `henrylavender`. Data pulled from 2004-01-01 to 2024-12-31.

---

## Daily Macro Data

### 5A: VIX Data
- **Table:** `cboe.cboe`
- **Variables:** OHLC for four CBOE volatility indices:
  - S&P 500 VIX: `vix`, `vixo`, `vixh`, `vixl`
  - S&P 100 VXO: `vxo`, `vxoo`, `vxoh`, `vxol`
  - NASDAQ VXN: `vxn`, `vxno`, `vxnh`, `vxnl`
  - DJIA VXD: `vxd`, `vxdo`, `vxdh`, `vxdl`

### 5B: Fama-French 5 Factors + Momentum
- **Table:** `ff_all.fivefactors_daily`
- **Variables:** `mktrf` (market excess return), `smb` (size), `hml` (value), `rmw` (profitability), `cma` (investment), `umd` (momentum), `rf` (risk-free rate)

### 5C: Daily FX Rates (11 Currencies)
- **Table:** `frb_all.fx_daily` (Federal Reserve Board daily FX dataset)
- **Currencies:** EUR, GBP, JPY, CHF, CAD, AUD, NOK, CNY, KRW, BRL, MXN
- All converted to "foreign currency units per 1 USD" convention. Direct quotes (`dex[XX]us`) are used as-is; inverted quotes (`dexus[XX]` for EUR and GBP) are flipped via 1/x.
- **FRED source columns:**
  - Direct: `dexjpus`, `dexszus`, `dexcaus`, `dexalus`, `dexnous`, `dexchus`, `dexkous`, `dexbzus`, `dexmxus`
  - Inverted: `dexuseu` (EUR), `dexusuk` (GBP)
- Rows where all FX columns are NaN (weekends/holidays in FRB calendar) are dropped.

### 5D: Daily World Index Returns (12 Countries)
- **Table:** `wrdsapps_windices.dwcountryreturns`
- **Countries:** JPN, CHN, KOR, AUS, HKG, GBR, DEU, FRA, CHE, BRA, IND, MEX
- Uses `portret` (cap-weighted equity index return with dividends). These are valuable because Asian and European markets close before the US opens, so their returns contain overnight information.
- Long-format data is pivoted to wide: one column per country, named `widx_jpn`, `widx_chn`, etc.

### Daily Merge
All four daily sources (VIX, FF5, FX, world indices) are outer-joined on `date`. Different sources have slightly different trading calendars, so NaN values for missing dates are expected.

---

## Monthly Macro Data

### 5F: US Treasury and Inflation
- **Table:** `crsp_a_indexes.mcti`
- **Variables:** All columns from the table, including bond returns and index levels at various maturities (1Y, 2Y, 5Y, 7Y, 10Y, 20Y, 30Y), T-bill returns (30-day, 90-day), and CPI data (`cpiind`, `cpiret`). Column `caldt` is renamed to `date`.

### 5G: Pastor-Stambaugh Liquidity Factors
- **Table:** `ff_all.liq_ps`
- **Variables:** `ps_level` (level of aggregate liquidity), `ps_innov` (innovation/surprise in aggregate liquidity). `ps_vwf` is excluded as it is entirely NULL.

### Monthly Merge
Treasury/CPI and Pastor-Stambaugh datasets are outer-joined on `date`.

---

## Outputs
- `Data/Data_Collection/Initial/09_Macro_Daily_Monthly_WRDS/macro_daily.parquet`
- `Data/Data_Collection/Initial/09_Macro_Daily_Monthly_WRDS/macro_monthly.parquet`

In [ ]:
# %% [markdown]
# # Stage 5: Collect All Macro / Market-Level Factor Data
#
# This notebook downloads all non-stock-specific data and saves it as two
# flat parquet files:
# - `data/macro_daily.parquet` — VIX, Fama-French 5, FX rates, world index returns
# - `data/macro_monthly.parquet` — US Treasury/CPI, Pastor-Stambaugh liquidity
#
# These datasets are small (a few thousand rows each) and require no PERMNO filtering.

# %% [markdown]
# ## Setup

# %%
import wrds
import pandas as pd
import numpy as np
from pathlib import Path


conn = wrds.Connection(wrds_username='henrylavender')

START = '2004-01-01'
END = '2024-12-31'

# %% [markdown]
# ---
# # DAILY MACRO DATA
# ---

# %% [markdown]
# ## 5A: VIX Data
#
# CBOE volatility indices — OHLC for S&P 500 (VIX), S&P 100 (VXO),
# NASDAQ (VXN), and DJIA (VXD). The S&P 500 VIX is the most important;
# the others provide cross-index vol information.

# %%
vix = conn.raw_sql(f"""
    SELECT date, vix, vixo, vixh, vixl,
           vxo, vxoo, vxoh, vxol,
           vxn, vxno, vxnh, vxnl,
           vxd, vxdo, vxdh, vxdl
    FROM cboe.cboe
    WHERE date >= '{START}' AND date <= '{END}'
""", date_cols=['date'])

print(f"VIX: {vix.shape} | {vix['date'].min().date()} to {vix['date'].max().date()}")
print(f"  VIX close range: {vix['vix'].min()} – {vix['vix'].max()}")

# %% [markdown]
# ## 5B: Fama-French 5 Factors + Momentum (Daily)
#
# The standard academic risk factors: market excess return, size (SMB),
# value (HML), profitability (RMW), investment (CMA), momentum (UMD),
# and the risk-free rate.

# %%
ff5 = conn.raw_sql(f"""
    SELECT date, mktrf, smb, hml, rmw, cma, rf, umd
    FROM ff_all.fivefactors_daily
    WHERE date >= '{START}' AND date <= '{END}'
""", date_cols=['date'])

print(f"FF5 daily: {ff5.shape} | {ff5['date'].min().date()} to {ff5['date'].max().date()}")
print(f"  mktrf range: {ff5['mktrf'].min():.4f} to {ff5['mktrf'].max():.4f}")

# %% [markdown]
# ## 5C: Daily FX Rates (11 selected currencies)
#
# We pull exchange rates (foreign currency units per 1 USD) from
# the Federal Reserve Board daily FX dataset on WRDS (frb_all.fx_daily).
#
# FRED column naming convention:
#   dex[XX]us = units of XX per 1 USD  (direct quote, use as-is)
#   dexus[XX] = units of USD per 1 XX  (inverted quote, need 1/x)
#
# Target currencies:
# - G4 majors: EUR, GBP, JPY, CHF
# - Commodity currencies: CAD, AUD, NOK
# - EM risk barometers: CNY, KRW, BRL, MXN

# %%
# Direct quotes: already "foreign currency per 1 USD"
fx_direct = {
    'fx_jpy': 'dexjpus',   # Japanese Yen per USD
    'fx_chf': 'dexszus',   # Swiss Franc per USD
    'fx_cad': 'dexcaus',   # Canadian Dollar per USD
    'fx_aud': 'dexalus',   # Australian Dollar per USD
    'fx_nok': 'dexnous',   # Norwegian Krone per USD
    'fx_cny': 'dexchus',   # Chinese Yuan per USD
    'fx_krw': 'dexkous',   # South Korean Won per USD
    'fx_brl': 'dexbzus',   # Brazilian Real per USD
    'fx_mxn': 'dexmxus',   # Mexican Peso per USD
}

# Inverted quotes: "USD per 1 foreign currency" → need 1/x
fx_invert = {
    'fx_eur': 'dexuseu',   # USD per Euro → invert
    'fx_gbp': 'dexusuk',   # USD per British Pound → invert
}

# Build column list for SQL query
all_fred_cols = list(fx_direct.values()) + list(fx_invert.values())
col_str_fred = ', '.join(['date'] + all_fred_cols)

fx_raw = conn.raw_sql(f"""
    SELECT {col_str_fred}
    FROM frb_all.fx_daily
    WHERE date >= '{START}' AND date <= '{END}'
""", date_cols=['date'])

print(f"FRB FX raw: {fx_raw.shape}")
print(f"Date range: {fx_raw['date'].min().date()} to {fx_raw['date'].max().date()}")

# %%
# Build the fx DataFrame with the same column names as the rest of the pipeline

fx = pd.DataFrame({'date': fx_raw['date']})

# Direct quotes — use as-is (convert to numeric to handle any NA types)
for target_col, fred_col in fx_direct.items():
    fx[target_col] = pd.to_numeric(fx_raw[fred_col], errors='coerce')

# Inverted quotes — flip to get foreign currency per USD
for target_col, fred_col in fx_invert.items():
    vals = pd.to_numeric(fx_raw[fred_col], errors='coerce')
    fx[target_col] = 1.0 / vals

# Drop rows where ALL fx columns are NaN (weekends/holidays in FRB calendar)
fx_cols = [c for c in fx.columns if c.startswith('fx_')]
fx = fx.dropna(subset=fx_cols, how='all').reset_index(drop=True)

print(f"\nFX wide: {fx.shape}")
print(f"Date range: {fx['date'].min().date()} to {fx['date'].max().date()}")

# Verify coverage
print(f"\nNaN counts per currency:")
for c in sorted(fx_cols):
    n = fx[c].isna().sum()
    pct = n / len(fx) * 100
    print(f"  {c}: {n:>5d} NaN ({pct:.1f}%)")

# Spot-check: last few rows to confirm values look reasonable
print(f"\nLast 3 rows (spot-check):")
print(fx.tail(3).to_string(index=False))

# %% [markdown]
# ## 5D: Daily World Index Returns (12 selected countries)
#
# Cap-weighted equity index returns (with dividends) for:
# - Asia-Pacific: JPN, CHN, KOR, AUS, HKG
# - Europe: GBR, DEU, FRA, CHE
# - EM: BRA, IND, MEX
#
# These are valuable because Asian and European markets close before the
# US opens, so their returns contain overnight information.
# We use `portret` (with dividends) only.

# %%
countries = ['JPN', 'CHN', 'KOR', 'AUS', 'HKG',
             'GBR', 'DEU', 'FRA', 'CHE',
             'BRA', 'IND', 'MEX']
country_str = ','.join(f"'{c}'" for c in countries)

widx_long = conn.raw_sql(f"""
    SELECT fic, date, portret
    FROM wrdsapps_windices.dwcountryreturns
    WHERE date >= '{START}' AND date <= '{END}'
      AND fic IN ({country_str})
""", date_cols=['date'])

print(f"World indices long: {widx_long.shape} | {widx_long['fic'].nunique()} countries")
print(f"  Countries found: {sorted(widx_long['fic'].unique())}")

# %%
# Pivot to wide: one column per country
widx = widx_long.pivot_table(
    index='date',
    columns='fic',
    values='portret',
    aggfunc='first'
).reset_index()

# Rename columns: JPN → widx_jpn, CHN → widx_chn, etc.
widx.columns = ['date'] + [f'widx_{c.lower()}' for c in widx.columns[1:]]

print(f"World indices wide: {widx.shape}")

# %% [markdown]
# ## 5E: Merge all daily sources
#
# Outer join on date — different sources have slightly different trading
# calendars (e.g., US markets open when Japan is on holiday). NaN values
# for missing dates are expected and acceptable.

# %%
macro_daily = vix.merge(ff5, on='date', how='outer')
macro_daily = macro_daily.merge(fx, on='date', how='outer')
macro_daily = macro_daily.merge(widx, on='date', how='outer')
macro_daily = macro_daily.sort_values('date').reset_index(drop=True)

print(f"Macro daily merged: {macro_daily.shape}")
print(f"Date range: {macro_daily['date'].min().date()} to {macro_daily['date'].max().date()}")

# %% [markdown]
# ### Save daily macro

# %%
macro_daily.to_parquet('../../Data/Data_Collection/Initial/09_Macro_Daily_Monthly_WRDS/macro_daily.parquet', index=False, engine='pyarrow')
print(f"Saved ../../Data/Data_Collection/Initial/09_Macro_Daily_Monthly_WRDS/macro_daily.parquet: {macro_daily.shape}")

# %% [markdown]
# ---
# # MONTHLY MACRO DATA
# ---

# %% [markdown]
# ## 5F: US Treasury and Inflation
#
# Bond returns and index levels at various maturities (1Y, 2Y, 5Y, 7Y,
# 10Y, 20Y, 30Y), T-bill returns (30-day, 90-day), and CPI data.
# All from `crsp_a_indexes.mcti`.

# %%
treasury = conn.raw_sql(f"""
    SELECT *
    FROM crsp_a_indexes.mcti
    WHERE caldt >= '{START}' AND caldt <= '{END}'
""", date_cols=['caldt'])

treasury = treasury.rename(columns={'caldt': 'date'})

print(f"Treasury/CPI: {treasury.shape} | {treasury['date'].min().date()} to {treasury['date'].max().date()}")
print(f"  Columns: {treasury.columns.tolist()}")

# %% [markdown]
# ## 5G: Pastor-Stambaugh Liquidity Factors
#
# Monthly market-level liquidity factors:
# - `ps_level` — level of aggregate liquidity
# - `ps_innov` — innovation (surprise) in aggregate liquidity
# - `ps_vwf` is all NULL so we drop it

# %%
ps = conn.raw_sql(f"""
    SELECT date, ps_level, ps_innov
    FROM ff_all.liq_ps
    WHERE date >= '{START}' AND date <= '{END}'
""", date_cols=['date'])

print(f"Pastor-Stambaugh: {ps.shape} | {ps['date'].min().date()} to {ps['date'].max().date()}")
print(f"  ps_level range: {ps['ps_level'].min():.4f} to {ps['ps_level'].max():.4f}")

# %% [markdown]
# ## 5H: Merge monthly sources

# %%
macro_monthly = treasury.merge(ps, on='date', how='outer')
macro_monthly = macro_monthly.sort_values('date').reset_index(drop=True)

print(f"Macro monthly merged: {macro_monthly.shape}")
print(f"Date range: {macro_monthly['date'].min().date()} to {macro_monthly['date'].max().date()}")

# %% [markdown]
# ### Save monthly macro

# %%
macro_monthly.to_parquet('../../Data/Data_Collection/Initial/09_Macro_Daily_Monthly_WRDS/macro_monthly.parquet', index=False, engine='pyarrow')
print(f"Saved ../../Data/Data_Collection/Initial/09_Macro_Daily_Monthly_WRDS/macro_monthly.parquet: {macro_monthly.shape}")

# %% [markdown]
# ---
# # VERIFICATION
# ---

# %% [markdown]
# ## Daily macro summary

# %%
print("=" * 70)
print("DAILY MACRO")
print("=" * 70)
print(f"Shape: {macro_daily.shape}")
print(f"Date range: {macro_daily['date'].min().date()} to {macro_daily['date'].max().date()}")
print(f"\nColumns ({len(macro_daily.columns)}):")
print(f"  {macro_daily.columns.tolist()}")

print(f"\nNull counts:")
nulls = macro_daily.isnull().sum()
nulls_nonzero = nulls[nulls > 0].sort_values(ascending=False)
if len(nulls_nonzero) > 0:
    for col, n in nulls_nonzero.items():
        print(f"  {col:25s} {n:5d} ({n/len(macro_daily)*100:5.1f}%)")
else:
    print("  No nulls!")

print(f"\nFirst 3 rows:")
print(macro_daily.head(3).to_string(index=False))
print(f"\nLast 3 rows:")
print(macro_daily.tail(3).to_string(index=False))

# %% [markdown]
# ### Sanity checks on value ranges

# %%
print("\nValue range sanity checks:")
print(f"  VIX close:  {macro_daily['vix'].min():.1f} – {macro_daily['vix'].max():.1f}  "
      f"(expect ~10-80)")
print(f"  mktrf:      {macro_daily['mktrf'].min():.4f} – {macro_daily['mktrf'].max():.4f}  "
      f"(expect ±0.10)")
print(f"  rf:         {macro_daily['rf'].min():.5f} – {macro_daily['rf'].max():.5f}  "
      f"(expect 0-0.0003)")

for curr in ['eur', 'gbp', 'jpy', 'cny']:
    col = f'fx_{curr}'
    if col in macro_daily.columns:
        vals = macro_daily[col].dropna()
        print(f"  {col}:  {vals.min():.2f} – {vals.max():.2f}")

# %% [markdown]
# ## Monthly macro summary

# %%
print("=" * 70)
print("MONTHLY MACRO")
print("=" * 70)
print(f"Shape: {macro_monthly.shape}")
print(f"Date range: {macro_monthly['date'].min().date()} to {macro_monthly['date'].max().date()}")
print(f"\nColumns ({len(macro_monthly.columns)}):")
print(f"  {macro_monthly.columns.tolist()}")

print(f"\nNull counts:")
nulls_m = macro_monthly.isnull().sum()
nulls_m_nonzero = nulls_m[nulls_m > 0].sort_values(ascending=False)
if len(nulls_m_nonzero) > 0:
    for col, n in nulls_m_nonzero.items():
        print(f"  {col:25s} {n:5d} ({n/len(macro_monthly)*100:5.1f}%)")
else:
    print("  No nulls!")

print(f"\nFirst 3 rows:")
print(macro_monthly.head(3).to_string(index=False))
print(f"\nLast 3 rows:")
print(macro_monthly.tail(3).to_string(index=False))

# %% [markdown]
# ### Sanity checks on monthly values

# %%
print("\nMonthly value range checks:")
print(f"  cpiind:     {macro_monthly['cpiind'].min():.1f} – {macro_monthly['cpiind'].max():.1f}  "
      f"(expect ~185-320)")
print(f"  cpiret:     {macro_monthly['cpiret'].min():.4f} – {macro_monthly['cpiret'].max():.4f}  "
      f"(expect ±0.02)")
print(f"  b10ret:     {macro_monthly['b10ret'].min():.4f} – {macro_monthly['b10ret'].max():.4f}  "
      f"(expect ±0.10)")
print(f"  ps_level:   {macro_monthly['ps_level'].min():.4f} – {macro_monthly['ps_level'].max():.4f}")
print(f"  ps_innov:   {macro_monthly['ps_innov'].min():.4f} – {macro_monthly['ps_innov'].max():.4f}")

# %% [markdown]
# ## Cleanup

# %%
conn.close()

print("\nStage 5 complete. Files saved:")
print("  Data/Data_Collection/Initial/09_Macro_Daily_Monthly_WRDS/macro_daily.parquet")
print("  Data/Data_Collection/Initial/09_Macro_Daily_Monthly_WRDS/macro_monthly.parquet")